In [11]:
%pip install pandas scikit-learn plotly pydeck joblib kagglehub streamlit

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import kagglehub
import glob
import os
import pandas as pd

# Tải dữ liệu từ Kaggle về máy
folder_path = kagglehub.dataset_download("toriqulstu/worlds-real-estate-data147k")

# Tự động tìm file CSV vừa tải
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
file_path = csv_files[0]

# Đọc dữ liệu vào bảng
df_raw = pd.read_csv(file_path)
print("✅ Đã nạp thành công dữ liệu thô! Số dòng:", len(df_raw))
df_raw.head(3)

✅ Đã nạp thành công dữ liệu thô! Số dòng: 147536


,title,country,location,building_construction_year,building_total_floors,apartment_floor,apartment_rooms,apartment_bedrooms,apartment_bathrooms,apartment_total_area,apartment_living_area,price_in_USD,image,url
0,2 room apartment 120 m² in Mediterranean Regio...,Turkey,"Mediterranean Region, Turkey",NaN,5.0,1.0,3.0,2.0,2.0,120 m²,110 m²,315209.0,https://realting.com/uploads/bigSlider/ab3/888...,https://realting.com/property-for-sale/turkey/...
1,"4 room villa 500 m² in Kalkan, Turkey",Turkey,"Kalkan, Mediterranean Region, Kas, Turkey",2021.0,2.0,NaN,NaN,NaN,NaN,500 m²,480 m²,1108667.0,https://realting.com/uploads/bigSlider/87b/679...,https://realting.com/property-for-sale/turkey/...
2,"1 room apartment 65 m² in Antalya, Turkey",Turkey,"Mediterranean Region, Antalya, Turkey",NaN,5.0,2.0,2.0,1.0,1.0,65 m²,60 m²,173211.0,https://realting.com/uploads/bigSlider/030/a11...,https://realting.com/property-for-sale/turkey/...


In [13]:
import numpy as np
import pandas as pd

df = df_raw.copy()
# 1. Đưa tất cả tên cột về chữ thường
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# In ra các cột thực tế để bạn quan sát
print("Các cột có trong dữ liệu gốc:", df.columns.tolist())

# 2. Tự động đổi tên các cột về chuẩn chung
for col in df.columns:
    if 'price' in col and 'price' not in df.columns:
        df = df.rename(columns={col: 'price'})
        print(f"-> Đã đổi cột '{col}' thành 'price'")
    elif ('location' in col or 'city' in col) and 'city' not in df.columns:
        df = df.rename(columns={col: 'city'})
        print(f"-> Đã đổi cột '{col}' thành 'city'")
    elif ('room' in col or 'bed' in col) and 'bedrooms' not in df.columns:
        df = df.rename(columns={col: 'bedrooms'})
        print(f"-> Đã đổi cột '{col}' thành 'bedrooms'")
    elif ('total_area' in col or col == 'area') and 'area' not in df.columns:
        df = df.rename(columns={col: 'area'})
        print(f"-> Đã đổi cột '{col}' thành 'area'")

# 3. Làm sạch cột giá (price)
df['price'] = df['price'].astype(str).str.replace(r'[\$,]', '', regex=True)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

# Bỏ dòng rỗng giá & lọc nhà có giá hợp lý ($10,000 đến $50,000,000)
df = df.dropna(subset=['price'])
df = df[(df['price'] >= 10000) & (df['price'] <= 50000000)]

# 4. Xử lý quốc gia & thành phố
if 'country' in df.columns:
    df['country'] = df['country'].fillna('United States').astype(str).str.strip().str.title()
else:
    df['country'] = 'Global'

if 'city' not in df.columns:
    df['city'] = 'Urban Area'
else:
    df['city'] = df['city'].fillna('Main City').astype(str).str.strip().str.title()

# 5. Xử lý số phòng & diện tích
if 'bedrooms' in df.columns:
    df['bedrooms'] = pd.to_numeric(df['bedrooms'], errors='coerce').fillna(2)
if 'area' in df.columns:
    df['area'] = pd.to_numeric(df['area'], errors='coerce').fillna(75.0)

# 6. Tọa độ để vẽ quả cầu 3D (Nếu thiếu thì gán theo tọa độ xấp xỉ của từng nước)
country_coords = {
    'United States': (37.0902, -95.7129), 'Canada': (56.1304, -106.3468),
    'United Kingdom': (55.3781, -3.4360), 'Australia': (-25.2744, 133.7751),
    'Vietnam': (14.0583, 108.2772), 'Germany': (51.1657, 10.4515),
    'France': (46.2276, 2.2137), 'Spain': (40.4637, -3.7492),
    'Japan': (36.2048, 138.2529), 'Georgia': (42.3154, 43.3569),
    'Turkey': (38.9637, 35.2433), 'Thailand': (15.8700, 100.9925)
}

if 'latitude' not in df.columns or 'longitude' not in df.columns:
    df['latitude'] = df['country'].map(lambda c: country_coords.get(c, (20.0, 0.0))[0]) + np.random.normal(0, 0.8, len(df))
    df['longitude'] = df['country'].map(lambda c: country_coords.get(c, (20.0, 0.0))[1]) + np.random.normal(0, 0.8, len(df))
else:
    df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce').fillna(20.0)
    df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce').fillna(0.0)

# 7. Lưu file dữ liệu sạch
df.to_csv("cleaned_real_estate.csv", index=False)
print("✅ THÀNH CÔNG! Đã tạo xong file 'cleaned_real_estate.csv' với", len(df), "bản ghi.")

Các cột có trong dữ liệu gốc: ['title', 'country', 'location', 'building_construction_year', 'building_total_floors', 'apartment_floor', 'apartment_rooms', 'apartment_bedrooms', 'apartment_bathrooms', 'apartment_total_area', 'apartment_living_area', 'price_in_usd', 'image', 'url']
-> Đã đổi cột 'location' thành 'city'
-> Đã đổi cột 'apartment_rooms' thành 'bedrooms'
-> Đã đổi cột 'apartment_total_area' thành 'area'
-> Đã đổi cột 'price_in_usd' thành 'price'
✅ THÀNH CÔNG! Đã tạo xong file 'cleaned_real_estate.csv' với 144169 bản ghi.


In [14]:
import joblib
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Lấy các đặc trưng
cat_cols = [c for c in ['country', 'property_type', 'type'] if c in df.columns]
num_cols = [c for c in ['bedrooms', 'bathrooms', 'area', 'sqft'] if c in df.columns]
features = cat_cols + num_cols

# Lấy 15,000 dòng đại diện để huấn luyện siêu nhanh
df_sample = df.sample(min(15000, len(df)), random_state=42)
X = df_sample[features]
y = np.log1p(df_sample['price'])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_cols),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols)
    ]
)

model_pipeline = Pipeline([
    ('prep', preprocessor),
    ('reg', HistGradientBoostingRegressor(max_iter=50, random_state=42))
])

model_pipeline.fit(X, y)

# Lưu mô hình AI ra file
joblib.dump(model_pipeline, "real_estate_model.pkl")
print("✅ Đã tạo xong file 'real_estate_model.pkl' thành công!")

✅ Đã tạo xong file 'real_estate_model.pkl' thành công!


In [15]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import pydeck as pdk
import joblib

st.set_page_config(page_title="Global Real Estate 3D", layout="wide", page_icon="🌍")

@st.cache_data
def load_data():
    return pd.read_csv("cleaned_real_estate.csv")

@st.cache_resource
def load_model():
    return joblib.load("real_estate_model.pkl")

df = load_data()
model = load_model()

st.title("🌍 Thị Trường Bất Động Sản Toàn Cầu - 3D Visualizer")

# --- 1. THANH TÌM KIẾM QUỐC GIA ---
all_countries = sorted(df['country'].dropna().unique().tolist())
selected_country = st.sidebar.selectbox("🔍 Chọn Quốc gia tìm kiếm:", ["Tất cả (Toàn cầu)"] + all_countries)

country_df = df if selected_country == "Tất cả (Toàn cầu)" else df[df['country'] == selected_country]

# Thống kê nhanh
col1, col2, col3 = st.columns(3)
col1.metric("Tổng Số Căn Nhà", f"{len(country_df):,}")
col2.metric("Giá Trung Bình", f"${country_df['price'].mean():,.0f}")
col3.metric("Giá Cao Nhất", f"${country_df['price'].max():,.0f}")

# --- 2. BẢN ĐỒ 3D GLOBE & ZOOM IN ---
st.subheader("🌐 Trực Quan Hóa Bản Đồ 3D")
if selected_country == "Tất cả (Toàn cầu)":
    # 3D Globe xoay tròn
    sample_df = country_df.sample(min(3000, len(country_df)), random_state=42)
    fig = go.Figure(go.Scattergeo(
        lon=sample_df['longitude'],
        lat=sample_df['latitude'],
        text=sample_df['city'] + ": $" + sample_df['price'].apply(lambda x: f"{x:,.0f}"),
        mode='markers',
        marker=dict(size=4, color=sample_df['price'], colorscale='Plasma', colorbar_title="Giá ($)")
    ))
    fig.update_geos(projection_type="orthographic", showcountries=True, showocean=True)
    fig.update_layout(height=550, margin={"r":0,"t":0,"l":0,"b":0})
    st.plotly_chart(fig, use_container_width=True)
else:
    # Pydeck 3D Zoom in vào quốc gia đó với các cột nổi 3D
    view = pdk.ViewState(
        latitude=country_df['latitude'].mean(), 
        longitude=country_df['longitude'].mean(), 
        zoom=5, 
        pitch=50
    )
    layer = pdk.Layer(
        "ColumnLayer", 
        country_df, 
        get_position=["longitude", "latitude"], 
        get_elevation="price", 
        elevation_scale=0.05, 
        radius=3000, 
        get_fill_color=[255, 120, 80, 180], 
        pickable=True
    )
    st.pydeck_chart(pdk.Deck(layers=[layer], initial_view_state=view, tooltip={"text": "{city}: ${price}"}))

# --- 3. BIỂU ĐỒ BAR CHART & PIE CHART ---
st.subheader("📊 Biểu Đồ Thống Kê & So Sánh Giá")
c1, c2 = st.columns(2)
with c1:
    top_cities = country_df.groupby('city')['price'].mean().reset_index().sort_values(by='price', ascending=False).head(10)
    fig_bar = px.bar(top_cities, x='city', y='price', title="Top Thành Phố Giá Cao Nhất", color='price', color_continuous_scale='Viridis')
    fig_bar.update_layout(xaxis_tickangle=-45)
    st.plotly_chart(fig_bar, use_container_width=True)
with c2:
    fig_pie = px.pie(country_df, names='country' if selected_country == "Tất cả (Toàn cầu)" else 'city', title="Phân Bổ Bất Động Sản", hole=0.35)
    st.plotly_chart(fig_pie, use_container_width=True)

# --- 4. DỰ ĐOÁN GIÁ BẰNG AI (ĐÃ BỔ SUNG CỘT AREA) ---
st.subheader("🤖 Dự Đoán Định Giá Bất Động Sản (Machine Learning)")
with st.form("predict_form"):
    p_col1, p_col2, p_col3 = st.columns(3)
    with p_col1:
        pred_country = st.selectbox("Chọn Quốc gia:", all_countries)
    with p_col2:
        pred_beds = st.number_input("Số phòng ngủ:", value=3, min_value=1, max_value=20)
    with p_col3:
        # Bổ sung ô nhập diện tích (area) ở đây
        pred_area = st.number_input("Diện tích căn hộ (m²):", value=75.0, min_value=10.0, max_value=2000.0, step=5.0)

    submit = st.form_submit_button("🚀 Dự Đoán Giá Ngay")

if submit:
    # Tự động lấy đúng tất cả các cột mà mô hình yêu cầu (tránh lỗi thiếu cột)
    required_cols = getattr(model, 'feature_names_in_', ['country', 'bedrooms', 'area'])
    input_data = {}
    for col in required_cols:
        if col == 'country':
            input_data[col] = [pred_country]
        elif col in ['bedrooms', 'apartment_rooms']:
            input_data[col] = [pred_beds]
        elif col in ['area', 'apartment_total_area', 'sqft']:
            input_data[col] = [pred_area]
        else:
            input_data[col] = [0]
            
    input_df = pd.DataFrame(input_data)
    
    # Dự đoán
    price_pred = np.expm1(model.predict(input_df)[0])
    st.success(f"💰 Mức giá ước tính cho căn hộ {pred_area} m² tại {pred_country}: **${price_pred:,.2f} USD**")

Overwriting app.py
